# CarveFormer — FFT-75 Benchmark (Kaggle Runner)

Full, top-to-bottom benchmark runner for **CarveFormer** (Swin Transformer V2
Tiny + 96-d byte embedding) on the FFT-75 dataset, using the shared **DeepCarv**
framework. This notebook is a benchmark runner, not a demo.

**Switch 512 <-> 4096 by changing `FRAGMENT_SIZE` in the config cell only.**

The FFT-75 dataset (a few GB) is **not** stored in the git repo. This notebook
downloads it from Google Drive with `gdown` at runtime.

Steps:
1. Install dependencies (incl. `gdown`)
2. Set paths, fragment size & the Google Drive ID
3. Download FFT-75 from Google Drive and extract it
4. Verify the NPZ layout
5. Sanity-train (few steps)
6. Full train
7. Evaluate
8. Save outputs
9. (optional) Compress the run

## 1. Install dependencies

In [ ]:
# timm provides the Swin Transformer V2 Tiny backbone. torch is preinstalled on
# Kaggle GPU images. gdown fetches the FFT-75 dataset from Google Drive.
!pip -q install "timm>=1.0.0" pyyaml gdown >/dev/null 2>&1
import torch, timm, gdown
print("torch", torch.__version__, "| timm", timm.__version__, "| gdown", gdown.__version__, "| cuda", torch.cuda.is_available())

## 2. Set paths and fragment size

This is the **only** cell you edit to switch block size or point at your data.

In [ ]:
import os
from pathlib import Path

# ---- The one switch: 512 or 4096 --------------------------------------------
FRAGMENT_SIZE = 512          # <-- change to 4096 for the 4 KiB benchmark
NUM_CLASSES   = 75           # FFT-75 Scenario #1 (11/25/5/2/2 for #2..#6)
PRETRAINED    = True         # ImageNet1k init (paper default)

# ---- Google Drive dataset source --------------------------------------------
# The FFT-75 dataset lives in Google Drive (too large for git). Paste ONE of:
#   * GDRIVE_FILE_ID   : the ID of a single .zip file containing FFT-75/, OR
#   * GDRIVE_FOLDER_ID : the ID of a shared Drive FOLDER holding the FFT-75 tree
# Get the ID from the Drive share link:
#   https://drive.google.com/file/d/<THIS_IS_THE_FILE_ID>/view
#   https://drive.google.com/drive/folders/<THIS_IS_THE_FOLDER_ID>
GDRIVE_FILE_ID   = ""        # e.g. "1AbCdEfGhIjKlMnOpQrStUvWxYz012345"
GDRIVE_FOLDER_ID = ""        # use this instead if the dataset is a shared folder

# ---- Paths ------------------------------------------------------------------
REPO_DIR   = Path("/kaggle/working/deepcarv")           # DeepCarv repo
DATA_ROOT  = Path("/kaggle/working/data")               # dataset download target
DATA_DIR   = DATA_ROOT / "FFT-75"                       # {DATA_DIR}/{fragment}/{train,val,test}.npz
WORK_DIR   = Path("/kaggle/working/carveformer_run")
for d in (DATA_ROOT, WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("fragment_size", FRAGMENT_SIZE, "| data_dir", DATA_DIR, "| work", WORK_DIR)

## 3. Download FFT-75 from Google Drive and bring in the repo

Uses `gdown` to pull the dataset from Drive into `DATA_DIR` (skipped if already
present, so re-runs are cheap). The benchmark reads the pre-split `.npz` files
directly — **no split regeneration, no CSV.** The DeepCarv repo is also cloned
so the framework + CarveFormer code are importable.

In [ ]:
import sys, zipfile, glob, shutil

# ---- 3a. Download the dataset from Google Drive (idempotent) ----------------
def _has_fft75(root: Path) -> bool:
    """True if the expected {root}/{size}/{split}.npz files already exist."""
    return (root / str(FRAGMENT_SIZE) / "train.npz").exists()

if _has_fft75(DATA_DIR):
    print("FFT-75 already present at", DATA_DIR, "- skipping download.")
elif GDRIVE_FILE_ID:
    # Single zip file -> download then extract.
    zip_path = DATA_ROOT / "fft75.zip"
    print("Downloading FFT-75 zip from Google Drive (file id)...")
    gdown.download(id=GDRIVE_FILE_ID, output=str(zip_path), quiet=False)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)
    # If the zip extracted to a differently-named folder, normalise it to FFT-75.
    if not _has_fft75(DATA_DIR):
        cands = [p for p in DATA_ROOT.iterdir()
                 if p.is_dir() and (p / str(FRAGMENT_SIZE) / "train.npz").exists()]
        if cands and cands[0] != DATA_DIR:
            if DATA_DIR.exists():
                shutil.rmtree(DATA_DIR)
            shutil.move(str(cands[0]), str(DATA_DIR))
    zip_path.unlink(missing_ok=True)
elif GDRIVE_FOLDER_ID:
    # Shared Drive folder -> download the whole tree into DATA_ROOT.
    print("Downloading FFT-75 folder from Google Drive (folder id)...")
    gdown.download_folder(id=GDRIVE_FOLDER_ID, output=str(DATA_DIR), quiet=False, use_cookies=False)
else:
    raise ValueError(
        "Set GDRIVE_FILE_ID or GDRIVE_FOLDER_ID in the config cell above "
        "so the dataset can be downloaded from Google Drive."
    )

assert _has_fft75(DATA_DIR), (
    f"Download finished but {DATA_DIR}/{FRAGMENT_SIZE}/train.npz is missing. "
    f"Check the Drive ID and the archive's internal folder structure."
)
print("FFT-75 ready at", DATA_DIR)

# ---- 3b. Bring in the DeepCarv repo (framework + CarveFormer code) -----------
if not REPO_DIR.exists():
    !git clone --branch eval-CarveFormer https://github.com/yuvnahr/deepcarv.git {REPO_DIR} 2>/dev/null || echo "clone skipped (provide repo manually)"
sys.path.insert(0, str(REPO_DIR))
print("repo on path:", REPO_DIR, "| exists:", REPO_DIR.exists())

## 4. Verify the expected NPZ layout

Confirm `{DATA_DIR}/{FRAGMENT_SIZE}/{train,val,test}.npz` exist and have the
expected arrays before spending GPU time.

In [ ]:
import numpy as np

frag_dir = DATA_DIR / str(FRAGMENT_SIZE)
for split in ("train", "val", "test"):
    p = frag_dir / f"{split}.npz"
    assert p.exists(), f"MISSING: {p}"
    with np.load(p) as d:
        keys = set(d.files)
        # The documented format is uppercase X/y; some loaders expect lowercase.
        xk = "X" if "X" in keys else ("x" if "x" in keys else None)
        yk = "y" if "y" in keys else None
        assert xk and yk, f"{p} keys {keys} — expected X/y"
        X, y = d[xk], d[yk]
        assert X.ndim == 2 and X.shape[1] == FRAGMENT_SIZE, f"{p} X shape {X.shape}"
        assert X.shape[0] == y.shape[0], f"{p} X/y length mismatch"
    print(f"OK  {split}: X={X.shape} y={y.shape} classes~{int(y.max())+1}")
print("NPZ layout verified.")

## 5. Sanity train (a couple of epochs on a subset)

Quick wiring check that config -> dataset -> registry -> Trainer -> Evaluator
all connect, before the full 50-epoch run.

In [ ]:
import yaml
from benchmarks.CarveFormer.scripts.train import run_training

# Load the committed benchmark config and apply notebook overrides.
CONFIG_PATH = REPO_DIR / "benchmarks/CarveFormer/configs/benchmark.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

config["dataset"]["fragment_size"] = FRAGMENT_SIZE
config["dataset"]["root_dir"]      = str(DATA_DIR)
config["dataset"]["num_classes"]   = NUM_CLASSES
config["model"]["pretrained"]      = PRETRAINED
config["paths"]["run_outputs"]     = str(WORK_DIR)

# Sanity: 2 epochs only.
sanity = {k: dict(v) if isinstance(v, dict) else v for k, v in config.items()}
sanity["training"] = dict(config["training"]); sanity["training"]["epochs"] = 2
run_training(sanity, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
print("Sanity pass complete.")

## 6. Full training (paper: 50 epochs, AdamW, lr 3.75e-4, wd 0.05)

The committed config already encodes the paper hyperparameters. Effective batch
size 1024 is documented in the config; adjust `batch_size` to your GPU.

In [ ]:
run_dir = run_training(config, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
print("Full training complete. Outputs:", run_dir)

## 7. Evaluate the best checkpoint

(The training call above already evaluates on the test split and writes the
standardized outputs; this cell re-runs evaluation standalone if desired.)

In [ ]:
from benchmarks.CarveFormer.scripts.evaluate import run_evaluation

best_ckpt = Path(WORK_DIR) / "checkpoint_best.pt"
if best_ckpt.exists():
    run_evaluation(config, best_ckpt, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
    print("Standalone evaluation complete.")
else:
    print("No checkpoint found; run training first.")

## 8. Inspect saved outputs

The framework writes the standardized set: `metrics.json`, `summary.json`,
`predictions.csv`, `confusion_matrix.csv`, `per_class_metrics.csv`,
`classification_report.txt`.

In [ ]:
import json
for name in ["summary.json", "metrics.json"]:
    p = Path(WORK_DIR) / name
    if p.exists():
        print("==", name, "==")
        print(json.dumps(json.load(open(p)), indent=2))
print("\nAll files:")
for p in sorted(Path(WORK_DIR).glob("*")):
    print(" ", p.name)

## 9. (Optional) Compress the run for download

In [ ]:
import shutil
archive = shutil.make_archive(f"/kaggle/working/carveformer_fft75_{FRAGMENT_SIZE}", "zip", WORK_DIR)
print("Archive:", archive)